# Notebook Details
- For the time being this is a pretty simple notebook setup to test the code for the accompanying RAG project at https://github.com/JoshuaAndle/Chatbot_Practice
- The notebook is designed to work with the accompanying code added to /content as /content/src/*

In [1]:
!pip install transformers
!pip install tokenizers
!pip install faiss-cpu
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 39.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [2]:
!unzip /content/src.zip -d /content/
!unzip /content/data.zip -d /content/

Archive:  /content/src.zip
   creating: /content/src/
  inflating: /content/src/configs.py  
   creating: /content/src/databases/
  inflating: /content/src/databases/database_manager.py  
  inflating: /content/src/databases/__init__.py  
  inflating: /content/src/main.py    
   creating: /content/src/models/
  inflating: /content/src/models/llm_models.py  
  inflating: /content/src/models/__init__.py  
  inflating: /content/src/__init__.py  
Archive:  /content/data.zip
   creating: /content/data/
  inflating: /content/data/huyen_research_abstracts.csv  
  inflating: /content/data/Sivesind_README.md  
  inflating: /content/data/sivesind_research_abstracts-final.csv  


# Setup the DataBases

In [5]:
dataset_name = "huyen_research_abstracts"
llm_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
embedding_model_name = "Qwen/Qwen3-Embedding-0.6B"
operation = "data_preparation"
db_batch_size = 32

!python3 "src/main.py" --operation=$operation --dataset_name=$dataset_name \
--llm_model_name=$llm_model_name --embedding_model_name=$embedding_model_name \
--db_batch_size=$db_batch_size --verbose

config.json: 100% 727/727 [00:00<00:00, 4.52MB/s]
tokenizer_config.json: 100% 9.71k/9.71k [00:00<00:00, 14.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 83.4MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 78.2MB/s]

tokenizer.json: downloading bytes:   0% 0.00/11.4M [00:00<?, ?B/s]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:01<00:00, 2.21MB/s,  330kB/s  ]
tokenizer.json: reconstructing file: 100% 11.4M/11.4M [00:01<00:00, 7.43MB/s, 1.11MB/s  ]

model.safetensors: downloading bytes:  13% 153M/1.19G [00:02<00:07, 134MB/s, 11.2MB/s  ] 
model.safetensors: downloading bytes:  18% 211M/1.19G [00:02<00:05, 191MB/s, 16.8MB/s  ]
model.safetensors: downloading bytes:  20% 240M/1.19G [00:03<00:05, 170MB/s, 19.8MB/s  ]
model.safetensors: downloading bytes:  22% 267M/1.19G [00:03<00:05, 158MB/s, 22.6MB/s  ]
model.safetensors: downloading bytes:  86% 1.03G/1.19G [00:06<00:01, 118MB/s, 74.3MB/s  ]
model.safetensors: reconstructing file:  26% 311M/1.19G [00:06<00:23, 37.4MB/s, 24.4MB

### Tests: Ensuring Proper Database Preparation
Just a quick check to make sure that all documents made it into the created databases

In [6]:
import pandas as pd
df = pd.read_parquet(f"./data/{dataset_name}.parquet")
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4053 entries, 0 to 4052
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   title                4053 non-null   object
 1   abstract             4053 non-null   object
 2   is_ai_generated      4053 non-null   int64 
 3   title_embeddings     4053 non-null   object
 4   abstract_embeddings  4053 non-null   object
dtypes: int64(1), object(4)
memory usage: 158.4+ KB
None


In [7]:
import faiss
loaded_index = faiss.read_index(f"./data/{dataset_name}.faiss")
print(f"Size of prepared FAISS Index for {dataset_name}: {loaded_index.ntotal}")

Size of prepared FAISS Index for huyen_research_abstracts: 4053


In [8]:
import chromadb

client = chromadb.PersistentClient(path=f"./data/chromadb_{dataset_name}")
if dataset_name == "huyen_research_abstracts":
    collection_name = "paper_abstracts"

collection = client.get_collection(collection_name)
print(f"Size of Resulting Chroma Collection: {collection.count()} \n {collection.peek()}")


Size of Resulting Chroma Collection: 4053 
 {'ids': ['paper_0', 'paper_1', 'paper_2', 'paper_3', 'paper_4', 'paper_5', 'paper_6', 'paper_7', 'paper_8', 'paper_9'], 'embeddings': array([[ 0.02355957, -0.04101563, -0.01062012, ..., -0.01623535,
        -0.05493164, -0.046875  ],
       [ 0.04077148, -0.01043701, -0.006073  , ...,  0.02355957,
        -0.02722168,  0.00939941],
       [ 0.06005859,  0.04516602, -0.00631714, ..., -0.02111816,
         0.02331543,  0.00726318],
       ...,
       [ 0.03710938,  0.00793457, -0.0057373 , ..., -0.05981445,
        -0.01062012,  0.01300049],
       [ 0.01660156,  0.0189209 , -0.00750732, ..., -0.03637695,
        -0.0189209 ,  0.01092529],
       [-0.04296875, -0.06689453, -0.00747681, ..., -0.00650024,
         0.02990723, -0.03857422]]), 'documents': ['  Human beings like to believe they are in control of their destiny. This\nubiquitous trait seems to increase motivation and persistence, and is probably\nevolutionarily adaptive. But how good 